## Modeling Pipeline

- **Baseline Model Development:** A standard model was trained using default/reasonable hyperparameters to establish a performance benchmark for each of the 6 algorithms (ARIMA, XGBoost, LightGBM, LSTM, CatBoost, Prophet).

- **Hyperparameter Optimization:** Each baseline model was optimized using two complementary techniques:
  - ***`Optuna`*** (Bayesian Optimization): Efficient, adaptive search with pruning for rapid convergence to high-performing configurations.
  - ***`RandomizedSearchCV`*** (Random Search with CV): Sklearn-compatible baseline optimization for fair comparison and reproducibility.
  - *Note: ARIMA and Prophet used model-specific optimization strategies due to API constraints.*

- **Cross-Validation and Robustness Assessment:** Each model variant was evaluated using ***`TimeSeriesSplit`*** (not random KFold) to preserve temporal order and prevent look-ahead bias. Metrics were aggregated across folds to assess stability.

- **Overfitting Analysis:** A detailed comparison between cross-validation metrics and test set results was conducted. Additional metrics, including ***`RMSE ratio`*** and ***`R² gap`***, were computed to quantify overfitting and assess model generalization. Directional accuracy and financial metrics (Sharpe Ratio, Max Drawdown) were also calculated for trading-relevant evaluation.

---

## Persisted Artifacts

To ensure reproducibility, transparency, and extendability, the following artifacts have been saved for **each model**:

- **Optimized Model Performance:** Individual CSV files capturing the performance of each model variant:
    - ***[Model] (Baseline)***
    - ***[Model] (Optuna)***
    - ***[Model] (RandomizedSearchCV)***

- **Best Variation Performance:** A CSV file containing only the metrics of the best-performing variation per model.

- **Summary of Model Performance:** A consolidated, extendable CSV file (`a_ModelPerformance.csv`) including:
    - Cross-validation results (`CV MSE`, `CV MAE`, `CV RMSE`, `CV R²`, `CV MAPE`)
    - Test set results (`Test MSE`, `Test MAE`, `Test RMSE`, `Test R²`, `Test MAPE`)
    - Financial metrics (`Sharpe Ratio`, `Sortino Ratio`, `Max Drawdown`, `Directional Accuracy`)
    - Overfitting metrics (`R² gap`, `RMSE ratio`)
    - Overfitting status and model generalization label

- **Overfitting DataFrame:** An extendable CSV (`a_OverfittingAnalysis.csv`) capturing overfitting analysis metrics across all models and variations.

- **Best Model per Algorithm:** The serialized best-performing variant of each algorithm for ensemble consideration or deployment.

- **Model Comparison Dashboard:** A summary notebook or script that loads `a_ModelPerformance.csv` and generates publication-ready comparison visualizations.

Together, these artifacts provide a complete, reproducible record of the modeling process, facilitating model tracking, comparison, selection, and deployment.

## Import Libraries and Root Configuration

In [1]:
""" Configure the utilities module path for imports """
import sys
import os
from pathlib import Path

# get project root as parent of current working directory
project_root = Path(os.getcwd()).parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [2]:
# records and calculations
import pandas as pd
import numpy as np

# avoid minor warnings
import warnings
warnings.filterwarnings('ignore')

# visualizations
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error
from scipy.stats import uniform, randint

# gradient boosting model
import xgboost as xgb

# optimization
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# import helper functions
from src.utilities import Evaluator, DataHandler, ModelPersister

## Load Dataset and Artifacts

In [3]:
df, x, y = DataHandler.load_dataset("../data/gold_price_engineered.csv", target_col="target")
artifacts = DataHandler.load_artifacts("../artifacts/FeatureSelection", cv_method='tscv')

In [4]:
# check dataset loading
df.head()

,Date,SPX_Return,GLD_Return,USO_Return,SLV_Return,EURUSD_Return,SPX_Return_lag1,SPX_Return_lag2,SPX_Return_lag3,SPX_Return_lag4,...,EURUSD_Return_lag2,EURUSD_Return_lag3,EURUSD_Return_lag4,EURUSD_Return_lag5,GLD_Return_lag1,GLD_Return_lag2,GLD_Return_lag3,GLD_Return_lag4,GLD_Return_lag5,target
0,2008-01-10,0.007948,0.019642,-0.016346,0.034858,0.009339,-0.002650,0.023711,-0.004229,-0.005142,...,0.023711,-0.004229,-0.005142,0.008367,-0.002650,0.023711,-0.004229,-0.005142,0.008367,0.003739
1,2008-01-11,-0.013595,0.003739,-0.012564,0.000996,-0.000739,0.019642,-0.002650,0.023711,-0.004229,...,-0.002650,0.023711,-0.004229,-0.005142,0.019642,-0.002650,0.023711,-0.004229,-0.005142,0.010838
2,2008-01-14,0.010871,0.010838,0.015871,0.012627,0.005337,0.003739,0.019642,-0.002650,0.023711,...,0.019642,-0.002650,0.023711,-0.004229,0.003739,0.019642,-0.002650,0.023711,-0.004229,-0.017311
3,2008-01-15,-0.024925,-0.017311,-0.019798,-0.027396,-0.004499,0.010838,0.003739,0.019642,-0.002650,...,0.003739,0.019642,-0.002650,0.023711,0.010838,0.003739,0.019642,-0.002650,0.023711,-0.014661
4,2008-01-16,-0.005612,-0.014661,-0.012778,-0.011368,-0.009326,-0.017311,0.010838,0.003739,0.019642,...,0.010838,0.003739,0.019642,-0.002650,-0.017311,0.010838,0.003739,0.019642,-0.002650,-0.002307


In [5]:
# check input features
x.head()

,Date,SPX_Return,GLD_Return,USO_Return,SLV_Return,EURUSD_Return,SPX_Return_lag1,SPX_Return_lag2,SPX_Return_lag3,SPX_Return_lag4,...,EURUSD_Return_lag1,EURUSD_Return_lag2,EURUSD_Return_lag3,EURUSD_Return_lag4,EURUSD_Return_lag5,GLD_Return_lag1,GLD_Return_lag2,GLD_Return_lag3,GLD_Return_lag4,GLD_Return_lag5
0,2008-01-10,0.007948,0.019642,-0.016346,0.034858,0.009339,-0.002650,0.023711,-0.004229,-0.005142,...,-0.002650,0.023711,-0.004229,-0.005142,0.008367,-0.002650,0.023711,-0.004229,-0.005142,0.008367
1,2008-01-11,-0.013595,0.003739,-0.012564,0.000996,-0.000739,0.019642,-0.002650,0.023711,-0.004229,...,0.019642,-0.002650,0.023711,-0.004229,-0.005142,0.019642,-0.002650,0.023711,-0.004229,-0.005142
2,2008-01-14,0.010871,0.010838,0.015871,0.012627,0.005337,0.003739,0.019642,-0.002650,0.023711,...,0.003739,0.019642,-0.002650,0.023711,-0.004229,0.003739,0.019642,-0.002650,0.023711,-0.004229
3,2008-01-15,-0.024925,-0.017311,-0.019798,-0.027396,-0.004499,0.010838,0.003739,0.019642,-0.002650,...,0.010838,0.003739,0.019642,-0.002650,0.023711,0.010838,0.003739,0.019642,-0.002650,0.023711
4,2008-01-16,-0.005612,-0.014661,-0.012778,-0.011368,-0.009326,-0.017311,0.010838,0.003739,0.019642,...,-0.017311,0.010838,0.003739,0.019642,-0.002650,-0.017311,0.010838,0.003739,0.019642,-0.002650


In [6]:
# check target feature
y.head()

0    0.003739
1    0.010838
2   -0.017311
3   -0.014661
4   -0.002307
Name: target, dtype: float64

In [7]:
# load train/test split data
x_train, x_test, y_train, y_test = artifacts['x_train'], artifacts['x_test'], artifacts['y_train'], artifacts['y_test']
cv = artifacts['cv']

# **Baseline Modeling**

In [8]:
# model config
BASELINE_PARAMS = {
    'objective': 'reg:squarederror',
    'n_estimators': 100,
    'learning_rate': 0.1,
    'max_depth' : 6,
    'random_state': 42,
    'verbosity': 0
}

# train baseline model
baseline_model = xgb.XGBRegressor(**BASELINE_PARAMS)
baseline_model.fit(x_train, y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


## Apply Model to Make Predication

In [9]:
# baseline prediction on train and test set
y_train_pred_baseline = baseline_model.predict(x_train)
y_test_pred_baseline = baseline_model.predict(x_test)

## Evaluating Model Performance
Calculating Errors for train and test data

In [10]:
# evaluation of train metrics
train_metrics_baseline = Evaluator.calculate_metrics(y_train, y_train_pred_baseline)

# evaluation of test metrics
test_metrics_baseline = Evaluator.calculate_metrics(y_test, y_test_pred_baseline)

In [11]:
# calculate directional time-series specific accuracy
train_acc_baseline = Evaluator.directional_accuracy(y_train, y_train_pred_baseline)
test_acc_baseline = Evaluator.directional_accuracy(y_test, y_test_pred_baseline)

In [12]:
# calcluate financial metrics of test data for baseline model
fin_baseline = Evaluator.financial_metrics("XGBoost_Baseline", y_test, y_test_pred_baseline)

display(fin_baseline)

,Model,Sharpe Ratio,Sortino Ratio,Max Drawdown,Total Return (%)
0,XGBoost_Baseline,-1.0071,-1.473,-30.2781,-19.2226


In [13]:
# baseline model performance
baseline_perf = Evaluator.performance_table(train_metrics_baseline + [train_acc_baseline], test_metrics_baseline + [test_acc_baseline])

print("XGBoost - Baseline Modeling Performance")
display(baseline_perf)

XGBoost - Baseline Modeling Performance


,Metrics,Training,Test
0,MSE,0.0001,0.0001
1,MAE,0.0056,0.0065
2,RMSE,0.0076,0.0087
3,R2 Score,0.6985,-0.1548
4,MAPE,15204.8711,40646.6552
5,Directional Accuracy (%),80.3943,47.9212


# **Optuna Hyperparameter Optimzation**

## **Find Best Parameters**

In [ ]:
# define objective function
def objective(trial):
    params = {
        "objective": "reg:squarederror",
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_weight": trial.suggest_float("min_child_weight", 1, 10),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-6, 10, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-6, 10, log=True),
        "random_state": 42,
        "verbosity": 0,
        "n_jobs": -1
    }

    # crate model
    model = xgb.XGBRegressor(**params)

    # cross validation with TimeSeriesSplit
    cv_scores = []
    for train_idx, val_idx in cv.split(x_train, y_train):
        x_tr, x_val = x_train.iloc[train_idx], x_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
 
        model.fit(x_tr, y_tr)

        y_pred = model.predict(x_val)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        cv_scores.append(rmse)

    return np.mean(cv_scores)

In [ ]:
# create study object
study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))

# optimize hyperparameters
study.optimize(objective, n_trials=50, show_progress_bar=False)

# best parameter from optuna
best_params_optuna = study.best_params

## Train optimized model and Apply Model to Make Predictions

In [ ]:
# train optimized model
optuna_model = xgb.XGBRegressor(**best_params_optuna, random_state=42, verbosity=0)
optuna_model.fit(x_train, y_train)

In [ ]:
# apply model to make predictions on train and test set
y_train_pred_optuna = optuna_model.predict(x_train)
y_test_pred_optuna = optuna_model.predict(x_test)

## Evaluating Optimized Model Performance
Calculating Errors for train and test data

In [ ]:
# calculate train and test metrics
# return [mse, mae, rmse, r2, mape]
train_metrics_optuna = Evaluator.calculate_metrics(y_train, y_train_pred_optuna)
test_metrics_optuna = Evaluator.calculate_metrics(y_test, y_test_pred_optuna)

In [ ]:
# performance table of optuna optimization
optuna_perf = Evaluator.performance_table(train_metrics_optuna, test_metrics_optuna)

print("XGBoost - Optuna Optimized Modeling Performance")
display(optuna_perf)

#  RandomizedSearchCV Optimzation

## Train RandomizedSearchCV Model

In [ ]:
# parameter distribution
param_dist = {
    "n_estimators": randint(100, 1000),
    "learning_rate": uniform(0.01, 0.29),
    "max_depth": randint(3, 13),
    "min_child_weight": uniform(1, 9),
    "subsample": uniform(0.6, 0.4),
    "colsample_bytree": uniform(0.6, 0.4),
    "gamma": uniform(0, 5),
    "reg_alpha": uniform(1e-6, 10),
    "reg_lambda": uniform(1e-6, 10),
}

# create empty model
model = xgb.XGBRegressor(
    objective='reg:squarederror',
    random_state=42,
    verbosity=0
)

In [ ]:
# optimize with RandomSearchCV
random_search = RandomizedSearchCV(model,
                                   param_dist,
                                   n_iter=50,
                                   scoring='neg_root_mean_squared_error',
                                   cv=cv,
                                   random_state=42,
                                   n_jobs=-1)

# fit model
random_search.fit(x_train, y_train)

# best model from RandomSearchCV
randomSearch_model = random_search.best_estimator_

## Apply Model to Make Predictions

In [ ]:
# make predictions on train and test set
y_train_pred_random = randomSearch_model.predict(x_train)
y_test_pred_random = randomSearch_model.predict(x_test)

## Evaluating Optimized Model Performance
Calculating Errors for train and test data

In [ ]:
# calculate train and test metrics
train_metrics_random = Evaluator.calculate_metrics(y_train, y_train_pred_random)
test_metrics_random = Evaluator.calculate_metrics(y_test, y_test_pred_random)

In [ ]:
# performance table of optuna optimization
random_perf = Evaluator.performance_table(train_metrics_random, test_metrics_random)

print("XGBoost - RandomSearchCV Optimized Modeling Performance")
display(random_perf)

# Cross Valiation (All Models)

In [ ]:
# cross validation of all models
cv_baseline = Evaluator.cv_evaluate(baseline_model, x_train, y_train, cv)
cv_optuna = Evaluator.cv_evaluate(optuna_model, x_train, y_train, cv)
cv_random = Evaluator.cv_evaluate(randomSearch_model, x_train, y_train, cv)

In [ ]:
# Evaluate cross-validation results
cv_df = pd.DataFrame({
    "Model": ["XGBoost (Baseline)", "XGBoost (Optuna)", "XGBoost (RandomSearchCV)"],
    "CV MSE": [cv_baseline['CV MSE'], cv_optuna['CV MSE'], cv_random['CV MSE']],
    "CV MAE": [cv_baseline['CV MAE'], cv_optuna['CV MAE'], cv_random['CV MAE']],
    "CV RMSE": [cv_baseline['CV RMSE'], cv_optuna['CV RMSE'], cv_random['CV RMSE']],
    "CV R2": [cv_baseline['CV R2'], cv_optuna['CV R2'], cv_random['CV R2']],
    "CV MAPE": [cv_baseline['CV MAPE'], cv_optuna['CV MAPE'], cv_random['CV MAPE']]
}).round(4)

print("Cross-Validation Results:")
display(cv_df)

# Summarize Models Performance

In [ ]:
# selected models
models = ['XGBoost (Baseline)', 'XGBoost (Optuna)', 'XGBoost (RandomSearchCV)']

# test metrics of all models
test_metrics = [test_metrics_baseline, test_metrics_optuna, test_metrics_random]

# performance summary including cross validation results
performance_summary = Evaluator.summary_builder(models, cv_df, test_metrics)

In [ ]:
# Display the final performance summary table
print("Overall Model Performance:")
display(performance_summary)

# Overfitting Analysis

In [ ]:
# Overfitting analysis of all models

rows = []
for _, row in performance_summary.iterrows():
    # Extract metrics needed for assess_overfitting
    cv_r2 = row['CV R2']
    test_r2 = row['Test R2']
    cv_rmse = row['CV RMSE']
    test_rmse = row['Test RMSE']
    
    # get overfitting and generalization status
    gap, rmse_ratio, overfit_status, gen_status = Evaluator.assess_overfitting(
        cv_r2=cv_r2,
        test_r2=test_r2,
        cv_rmse=cv_rmse,
        test_rmse=test_rmse,
        tolerance=0.05
    )
    
    # build row
    rows.append({
        "Model": row['Model'],
        "CV RMSE": row['CV RMSE'],
        "CV R2": row['CV R2'],
        "Test RMSE": row['Test RMSE'],
        "Test R2": row['Test R2'],
        "R2 Gap": gap,
        "RMSE Ratio": rmse_ratio,
        "Overfitting Status": overfit_status,
        "Model Status (Generalization)": gen_status
    })

overfit_df = pd.DataFrame(rows).round(4)

In [ ]:
print("XGBoost - Overfitting Analysis:")
display(overfit_df)

# Model Comparison (Visualization)

In [ ]:
""" XGBoost Model Performance Comparison """

# Define metrics and titles
metrics = [
    "Test R2",
    "Test RMSE",
    "RMSE Ratio",
    "Test MAPE"
]

titles = [
    "Test R² (↑ better)",
    "Test RMSE (↓ better)",
    "Test RMSE Ratio (↓ better)",
    "Test MAPE (%) (↓ better)"
]

# Data preparation
model_labels = ["Baseline", "Optuna", "RandomSearch"]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

values_dict = {
    "Test R2": overfit_df["Test R2"].values,
    "Test RMSE": overfit_df["Test RMSE"].values,
    "RMSE Ratio": overfit_df["RMSE Ratio"].values,
    "Test MAPE": [test_metrics_baseline[4], test_metrics_optuna[4], test_metrics_random[4]]
}

# Figure setup
n_metrics = len(metrics)
rows, cols = 2, 2

fig, axes = plt.subplots(rows, cols, figsize=(14, 10))
axes = axes.flatten()

# Plot each metric dynamically
for i, metric in enumerate(metrics):
    ax = axes[i]
    values = values_dict[metric]
    
    # Create horizontal bar chart with model-specific colors
    bars = ax.bar(model_labels, values, color=colors)
    
    # Title and labels
    ax.set_title(titles[i], fontsize=12)
    

# Hide any unused subplots (if any)
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

# Add overall title and adjust layout
plt.suptitle("XGBoost Model Comparison Across Metrics", fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout(pad=3.0, h_pad=3, w_pad=5)
plt.show()


# Final Summary of Model Performances

In [ ]:
# final summary of all models performance
final_summary = pd.merge(performance_summary,
                    overfit_df[['Model', 'R2 Gap', 'RMSE Ratio', 'Overfitting Status', 'Model Status (Generalization)']],
                    on="Model",
                    how="outer")

In [ ]:
print("Summary of The Models Performance:")
display(final_summary)

# Persist Best Model and Performance

In [ ]:
# select best variant of XGBoost model
best_idx = final_summary["Test R2"].idxmax()
best_variant = final_summary.iloc[best_idx]

In [ ]:
print("Best Variation of XGBoost Model:", best_variant["Model"])

In [ ]:
# performance of best variation
best_variant = best_variant.to_frame(name='Value').rename_axis('Metrics').reset_index()

print("Performance of the best variation:")
display(best_variant)

In [ ]:
# model persister object
persister = ModelPersister(model_name = "XGBoost")

In [ ]:
# persit all variatnt performance
persister.save_performance(performance_summary)

In [ ]:
# persist best variant performance
persister.save_performance(best_variant, "BestVariation")

In [ ]:
# aggregated model performance
persister.aggregated_performance(final_summary)

In [ ]:
# save overfitting overfitting analysis
persister.append_overfitting(overfit_df)

In [ ]:
# save the best variant of XGBoost model
persister.save_model(randomSearch_model)